In [ ]:
import random
import tkinter as tk
from tkinter import messagebox

class Card:
    def __init__(self, color, value):
        self.color = color
        self.value = value
    def __repr__(self):
        return f"{self.color} {self.value}"

def generate_deck():
    colors = ["Red", "Blue", "Green", "Yellow"]
    deck = []
    for color in colors:
        for num in range(10):
            deck.append(Card(color, num))
        deck.append(Card(color, "Skip"))
    random.shuffle(deck)
    return deck

def init_game():
    deck = generate_deck()
    return {
        "p1_hand": [deck.pop() for i in range(5)],   #5 cards per player
        "p2_hand": [deck.pop() for i in range(5)],
        "p3_hand": [deck.pop() for i in range(5)],
        "top_card": deck.pop(),
        "deck": deck,
        "current_player": 1,
        "skip_next": False
    }

def get_valid_moves(hand, top_card):
    moves = []
    for card in hand:
        #card is valid if Color matches or Value matches
        if card.color == top_card.color or str(card.value) == str(top_card.value):
            moves.append(card)
    return moves

def apply_move(state, player, move):
    new_state = {k: (v[:] if isinstance(v, list) else v) for k, v in state.items()}
    hand = new_state[f"p{player}_hand"]
    
    if move == "DRAW":
        if new_state["deck"]: hand.append(new_state["deck"].pop())  #draw wala card
    else:
        for c in hand:
            if c.color == move.color and c.value == move.value:
                hand.remove(c)
                break
        new_state["top_card"] = move
        if move.value == "Skip": new_state["skip_next"] = True

    next_p = ((player) % 3) + 2 if new_state["skip_next"] else (player % 3) + 1
    if next_p > 3: next_p -= 3
    new_state["current_player"] = next_p
    new_state["skip_next"] = False
    return new_state

def is_terminal(state):
    return any(len(state[f"p{i}_hand"]) == 0 for i in range(1, 4))  # end if p has 0 cards

def evaluate(state, player_id, mode):
    ai_h = len(state[f"p{player_id}_hand"])
    others = [len(state[f"p{i}_hand"]) for i in range(1, 4) if i != player_id]
    opp_avg = sum(others) / 2
    skips = sum(1 for c in state[f"p{player_id}_hand"] if c.value == "Skip")
    
    if mode == "defensive":
        return 60 - 5*ai_h - 6*(10 - opp_avg) + 5*skips
    else:
        return 60 - 9*ai_h + 1*opp_avg + 2*skips

def minimax(state, depth, player_id):
    if depth == 0 or is_terminal(state):
        return evaluate(state, player_id, "defensive")
    curr = state["current_player"]
    moves = get_valid_moves(state[f"p{curr}_hand"], state["top_card"]) or ["DRAW"]
    
    if curr == player_id:  # maximize score
        return max(minimax(apply_move(state, curr, m), depth-1, player_id) for m in moves)
    else:
        return min(minimax(apply_move(state, curr, m), depth-1, player_id) for m in moves) #minimize score

def expectimax(state, depth):
    if depth == 0 or is_terminal(state):
        return evaluate(state, 2, "offensive")
    curr = state["current_player"]
    moves = get_valid_moves(state[f"p{curr}_hand"], state["top_card"]) or ["DRAW"]
    
    if curr == 2: #best move of ai
        return max(expectimax(apply_move(state, curr, m), depth-1) for m in moves)
    else: #opp acts random
        return sum(expectimax(apply_move(state, curr, m), depth-1) for m in moves) / len(moves)

class UnoGUI:
    def __init__(self, root):
        self.root = root
        self.root.title("UNO AI Strategy Engine")
        self.root.geometry("850x750")
        
        self.is_sim = messagebox.askyesno("Mode Selection", "Run as full Simulation (AI vs AI)?")
        self.state = init_game()

        tk.Label(root, text="CURRENT TOP CARD", font=("Arial", 10)).pack(pady=5)
        self.top_display = tk.Label(root, text="", font=("Arial", 22, "bold"), fg="white", width=15, height=3)
        self.top_display.pack(pady=10)

        self.info_label = tk.Label(root, text="", font=("Courier", 12, "bold"))
        self.info_label.pack()

        self.log_box = tk.Text(root, height=12, width=85)
        self.log_box.pack(pady=10)

        self.hand_frame = tk.Frame(root)
        self.hand_frame.pack(pady=10)

        self.btn_next = tk.Button(root, text="EXECUTE AI TURN", bg="orange", font=("Arial", 12, "bold"), 
                                  command=self.play_ai_step, width=25, height=2)
        self.btn_next.pack(pady=10)

        self.update_ui()

    def update_ui(self):
        tc = self.state["top_card"]
        self.top_display.config(text=str(tc), bg=tc.color.lower())
        stats = f"P1: {len(self.state['p1_hand'])} | P2: {len(self.state['p2_hand'])} | P3: {len(self.state['p3_hand'])}"
        self.info_label.config(text=stats)  # showing player kai card ka cont

        for w in self.hand_frame.winfo_children(): w.destroy()  # clearing old buttons
        
        if self.state["current_player"] == 3 and not self.is_sim: #player3
            self.btn_next.config(state="disabled", text="YOUR TURN")
            valid = get_valid_moves(self.state["p3_hand"], self.state["top_card"])
            if not valid:
                tk.Button(self.hand_frame, text="DRAW", bg="black", fg="white", command=lambda: self.apply_and_next("DRAW")).pack()
            else:
                for c in self.state["p3_hand"]:
                    cmd = (lambda x=c: self.apply_and_next(x)) if c in valid else None
                    tk.Button(self.hand_frame, text=str(c), bg=c.color.lower(), fg="white", 
                              state=("normal" if cmd else "disabled"), command=cmd).pack(side="left", padx=2)
        else:
            self.btn_next.config(state="normal", text=f"EXECUTE P{self.state['current_player']} TURN")

    def apply_and_next(self, move):
        self.log(f"P3 plays: {move}")
        self.state = apply_move(self.state, 3, move)
        self.update_ui()
        self.check_win()

    def play_ai_step(self):
        p = self.state["current_player"]
        if p == 3 and not self.is_sim: return

        self.log(f"\nP{p} thinking...")
        hand = self.state[f"p{p}_hand"]
        moves = get_valid_moves(hand, self.state["top_card"])
        
        if not moves:
            move = "DRAW"
        else:
            best_m = moves[0]
            best_s = -9999
            # Log output formatting to match sample
            score_type = "Expected Score" if p == 2 else "Score"
            
            for m in moves:
                score = minimax(apply_move(self.state, p, m), 2, p) if p != 2 else expectimax(apply_move(self.state, p, m), 2)
                self.log(f"Play {m} -> {score_type}: {round(score, 2)}")
                if score > best_s:
                    best_s, best_m = score, m
            move = best_m

        self.log(f"Chosen: {move}")
        self.state = apply_move(self.state, p, move)
        self.update_ui()
        self.check_win()

    def log(self, msg):
        self.log_box.insert(tk.END, msg + "\n")
        self.log_box.see(tk.END)
        print(msg)

    def check_win(self):
        for i in range(1, 4):
            if len(self.state[f"p{i}_hand"]) == 0:
                messagebox.showinfo("Game Over", f"Player {i} wins!")
                self.root.destroy()

if __name__ == "__main__":
    root = tk.Tk()
    app = UnoGUI(root)
    root.mainloop()